# QM 640 Capstone — Step 3: Manual Screening & Announcement-Type Classification

This step is deliberately NOT automated end-to-end — the Synopsis's Data
Quality Risk section requires manual review of 8-K text and newswire
language to (a) apply the exclusion criteria and (b) classify
`announcement_type`. This notebook builds the worksheet and, later, checks
inter-rater reliability. **The actual screening happens outside this
notebook, in a spreadsheet.**

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 899, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 899 (delta 37), reused 51 (delta 19), pack-reused 813 (from 1)
Receiving objects: 100% (899/899), 5.44 MiB | 11.03 MiB/s, done.
Resolving deltas: 100% (478/478), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas scikit-learn

## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

## Part A — Build the worksheet (run once)

Converts the raw EDGAR candidate pool into a screening worksheet with blank
columns for manual review, plus a 20% random subsample set aside for
independent re-coding.

In [ ]:
import pandas as pd
import csv


def build_worksheet():
    src = os.path.join(RAW_DIR, "edgar_candidate_events.csv")
    df = pd.read_csv(src)

    df["file_date"] = pd.to_datetime(df["file_date"])
    df = df.sort_values("file_date").drop_duplicates(subset=["cik", "file_date"], keep="first")

    df["is_genuine_ai_event"] = ""       # Y/N - excludes incidental AI mentions
    df["announcement_type"] = ""          # partnership / R&D / M&A
    df["confounding_event_flag"] = ""     # Y/N - other material event in [-2,+2]
    df["trading_halt_flag"] = ""          # Y/N
    df["sufficient_history_flag"] = ""    # Y/N - >=120 trading days pre-event
    df["exclude_reason"] = ""             # free text if excluded
    df["filing_url"] = df.apply(
        lambda r: f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={r['cik']}",
        axis=1,
    )
    df["screener_notes"] = ""

    df.to_csv(SCREENING_FILE, index=False, quoting=csv.QUOTE_ALL)
    print(f"Worksheet built: {len(df)} candidate events -> {SCREENING_FILE}")

    n_subsample = max(1, int(len(df) * 0.20))
    subsample = df.sample(n=n_subsample, random_state=42)
    subsample_out = subsample[["accession_no", "company_name", "file_date"]].copy()
    subsample_out["recoder_announcement_type"] = ""
    subsample_out["recoder_is_genuine_ai_event"] = ""
    subsample_out.to_csv(RECODE_FILE, index=False, quoting=csv.QUOTE_ALL)
    print(f"20% re-coding sample ({n_subsample} events) -> {RECODE_FILE}")
    return df


worksheet = build_worksheet()
worksheet.head()

Worksheet built: 9419 candidate events -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv
20% re-coding sample (1883 events) -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_recode_sample.csv


,query,cik,company_name,form_type,file_date,accession_no,adsh,file_name,is_genuine_ai_event,announcement_type,confounding_event_flag,trading_halt_flag,sufficient_history_flag,exclude_reason,filing_url,screener_notes
8855,"""AI capabilities""",876167,PROGRESS SOFTWARE CORP /MA (PRGS) (CIK 00008...,8-K,2023-01-03,0000876167-23-000004:pressrelease-marklogic.htm,0000876167-23-000004,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
3872,"""AI-powered""",1013857,PEGASYSTEMS INC (PEGA) (CIK 0001013857),8-K,2023-01-03,0001193125-23-000843:d442682dex991.htm,0001193125-23-000843,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
4451,"""AI-powered""",1293818,OPGEN INC (OPGN) (CIK 0001293818),8-K,2023-01-04,0001079973-23-000010:ex99x1.htm,0001079973-23-000010,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
10460,"""AI-based""",278165,OMNIQ Corp. (OMQS) (CIK 0000278165),8-K,2023-01-04,0001493152-23-000229:ex99-1.htm,0001493152-23-000229,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,
6481,"""AI-driven""",1758766,"STEM, INC. (STEM) (CIK 0001758766)",8-K,2023-01-05,0001758766-23-000003:exhibit99-1320238xkgsconf...,0001758766-23-000003,NaN,,,,,,,https://www.sec.gov/cgi-bin/browse-edgar?actio...,


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} add "data/raw/screening_recode_sample.csv"
!git -C {BASE_DIR} commit -m "Step 3a: build screening worksheet + recode sample"
!git -C {BASE_DIR} push

[main ac42a30] Step 3a: build screening worksheet + recode sample
 2 files changed, 11304 insertions(+)
 create mode 100644 data/raw/screening_recode_sample.csv
 create mode 100644 data/raw/screening_worksheet.csv
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (6/6), 428.23 KiB | 2.73 MiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   0895cdf..ac42a30  main -> main


## >>> STOP HERE — manual step, outside this notebook <<<

1. Download `screening_worksheet.csv` (or edit it directly on GitHub / in
   Google Sheets after downloading) and manually review each row against
   the exclusion criteria in the Synopsis, using the `filing_url` link to
   read the actual 8-K text.
2. Fill in `is_genuine_ai_event`, `announcement_type`,
   `confounding_event_flag`, `trading_halt_flag`, `sufficient_history_flag`.
3. Re-upload the completed CSV back into `data/raw/screening_worksheet.csv`
   in the repo (via `git push` from your local machine, GitHub's web upload,
   or by re-running Cell 1 in a fresh Colab session, replacing the file, and
   running the push cell below).
4. Give `screening_recode_sample.csv` to an independent reviewer, **without
   your own classifications visible**, and have them fill in
   `recoder_announcement_type` / `recoder_is_genuine_ai_event`. Merge that
   back into the repo the same way.

Once both files are updated in the repo, continue to Part B below.